<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

## 02-1 · Part 1: Basic Optimization Formulation

**We want to choose early and late cooling levels that keep the classroom acceptable while balancing comfort and energy use.**

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="assets/01_three_lecture_bridge.svg" alt="Three connected stages from the classroom system model to a standard optimization formulation" width="760" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

Lecture 01-1 supplied the state equation. Lecture 01-2 identified the decision, performance outputs, objective, and requirements. This lecture gives those parts their standard optimization names without changing their physical meaning.

A **system model** predicts what happens after a decision. An **optimization formulation** states the decision variables, decision domain, objective, constraints, and system model together.


### 1 · Start from the existing classroom decision

The physical decision remains

> $\displaystyle u=(u_{\mathrm{early}},u_{\mathrm{late}}).$

It expands into 12 cooling actions:

> $\displaystyle u_t=\begin{cases}u_{\mathrm{early}},&t=0,\ldots,5,\\u_{\mathrm{late}},&t=6,\ldots,11.\end{cases}$

The state equation from 01-1 remains unchanged:

> $\displaystyle T_{t+1}=T_t+a\left(T_t^{\mathrm{out}}-T_t\right)+bN_t-cu_t,\qquad t=0,\ldots,n-1,$

with \(n=12\), \(T_0=27\,^\circ\mathrm C\), \(T_t^{\mathrm{out}}=31\,^\circ\mathrm C\), \(N_t=20\), and \((a,b,c)=(0.12,0.012,0.45)\). A decision changes the real cooling action; the temperature path is produced by the system.

The code below keeps the fixed parameters, external inputs, simulation, performance calculation, feasibility test, and score visibly separated.


In [ ]:
import numpy as np

# Horizon and initial state
TIME_STEPS = 12
INITIAL_TEMPERATURE = 27.0

# Fixed parameters
WEATHER_EXCHANGE = 0.12
OCCUPANT_HEAT = 0.012
COOLING_EFFECT = 0.45

# External inputs
OUTSIDE_TEMPERATURE = np.full(TIME_STEPS, 31.0)
OCCUPANTS = np.full(TIME_STEPS, 20.0)

# Decision domain and requirement limits
MIN_COOLING, MAX_COOLING = 0.0, 5.0
MIN_TEMPERATURE, MAX_TEMPERATURE = 20.0, 30.0
MAX_ENERGY = 60.0


def expand_decision(x):
    early_cooling, late_cooling = np.asarray(x, dtype=float)
    return np.r_[np.full(6, early_cooling), np.full(6, late_cooling)]


def simulation_model(x):
    cooling_schedule = expand_decision(x)
    temperatures = [INITIAL_TEMPERATURE]
    for outdoor, people, cooling in zip(
        OUTSIDE_TEMPERATURE, OCCUPANTS, cooling_schedule
    ):
        current = temperatures[-1]
        temperatures.append(
            current
            + WEATHER_EXCHANGE * (outdoor - current)
            + OCCUPANT_HEAT * people
            - COOLING_EFFECT * cooling
        )

    temperatures = np.asarray(temperatures)
    discomfort = np.sum(
        np.maximum(temperatures[1:] - 24.0, 0.0) ** 2
        + np.maximum(22.0 - temperatures[1:], 0.0) ** 2
    )
    energy = 0.5 * np.sum(cooling_schedule**2)
    return {
        "temperatures": temperatures,
        "discomfort": float(discomfort),
        "energy": float(energy),
    }


def objective_function(y, energy_weight=1.0):
    return y["discomfort"] + energy_weight * y["energy"]


def inequality_constraints(x, y):
    early_cooling, late_cooling = np.asarray(x, dtype=float)
    temperatures = y["temperatures"][1:]
    return {
        "early lower": MIN_COOLING - early_cooling,
        "early upper": early_cooling - MAX_COOLING,
        "late lower": MIN_COOLING - late_cooling,
        "late upper": late_cooling - MAX_COOLING,
        "temperature lower": MIN_TEMPERATURE - temperatures.min(),
        "temperature upper": temperatures.max() - MAX_TEMPERATURE,
        "energy": y["energy"] - MAX_ENERGY,
    }


def evaluate_formulation(x, energy_weight=1.0):
    x = tuple(map(float, x))
    y = simulation_model(x)
    g = inequality_constraints(x, y)
    return {
        "x": x,
        "y": y,
        "f": objective_function(y, energy_weight),
        "g": g,
        "feasible": all(value <= 1e-10 for value in g.values()),
    }


### 2 · Give each part its standard name

The classroom symbols and the general symbols refer to the same objects:

| General symbol | Meaning | Classroom instance |
|:---:|:---|:---|
| $x$ | Decision vector | $[u_{\mathrm{early}},u_{\mathrm{late}}]^{\mathsf{T}}$ |
| $\mathcal X$ | Allowed decision domain | $[0,5]^2$ |
| $y=\operatorname{Sim}(x)$ | Responses produced by simulation | $[T_1(u),\ldots,T_n(u),D(u),E(u)]^{\mathsf{T}}$ |
| $f(y;\lambda_E)$ | Scalar comparison rule | $J(u;\lambda_E)=D(u)+\lambda_EE(u)$ |
| $g_j(x,y)$ | Inequality residual | Cooling, temperature, and energy residuals |
| $h_k(x,y)$ | Equality residual | State equations when states are explicit variables |

The simulator repeats the physical transition \(F\) and then applies the performance mapping \(G\) to produce \(y\). The objective \(f\) is the standard-form name for the score supplied by \(H\). Uppercase \(F\) remains the physical state transition.

The complete simulation-based formulation answers one question: which allowed decision gives the lowest score while satisfying every requirement?

> $\displaystyle \underset{x\in\mathcal X}{\operatorname{minimize}}\quad f(y)$
>
> $\displaystyle \text{subject to}\quad y=\operatorname{Sim}(x)$
>
> $\displaystyle g_j(x,y)\le0\quad(j=1,\ldots,m_g)$
>
> $\displaystyle h_k(x,y)=0\quad(k=1,\ldots,m_h)$

Here, \(m_g\) and \(m_h\) are the numbers of inequality and equality constraints. A hyperparameter such as \(\lambda_E\) is fixed during one optimization run.

The next calculation follows one candidate through every part of the formulation.


In [ ]:
candidate = evaluate_formulation((3.0, 2.0), energy_weight=1.0)

print(f"x = {candidate['x']}")
print(f"temperature range = {candidate['y']['temperatures'][1:].min():.2f} to "
      f"{candidate['y']['temperatures'][1:].max():.2f} °C")
print(f"D = {candidate['y']['discomfort']:.2f}")
print(f"E = {candidate['y']['energy']:.2f}")
print(f"f = J = {candidate['f']:.2f}")
print(f"all g_j <= 0: {candidate['feasible']}")


### 3 · Read a formulation in causal order

For \(x=[3,2]^{\mathsf{T}}\), the simulator first produces the temperature path. The performance mapping then produces discomfort \(D\) and energy \(E\). The constraints determine eligibility, and the objective supplies the comparison value.

> **choose \(x\) → simulate \(y\) → check \(g,h\) → compare feasible candidates with \(f\)**

Feasibility comes before objective comparison. An infeasible candidate is rejected regardless of its score.

A formulation defines **what problem is being solved**. Grid search, gradient descent, and particle swarm optimization are algorithms that describe **how candidates are searched**. Changing the algorithm does not change the formulation.

The remaining 02-1 notebooks hold most of this formulation fixed and change one part at a time. That controlled comparison reveals which optimization class results from each change.


### Takeaway

A complete optimization formulation connects the real decision to a mathematical selection rule:

> **decision \(x\) and domain \(\mathcal X\) → responses \(y=\operatorname{Sim}(x)\) → feasibility from \(g,h\) → comparison by \(f\)**

For the classroom, \(x\) is the column-vector form of \(u=(u_{\mathrm{early}},u_{\mathrm{late}})\), \(y\) contains \(T_1,\ldots,T_n,D,E\), and \(f(y;\lambda_E)=J(u;\lambda_E)\).

Part 2 keeps the decision, simulator, and objective fixed and changes only the constraints.
